In [5]:
import random
import soundfile as sf  

# imports
import pandas as pd
import numpy as np
import librosa
from pathlib import Path
from scipy.fft import rfft
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy
from scipy import signal
import noisereduce as nr
import math 
import os
import cv2

import shutil

In [6]:
# Directories to sample from
roots = [
    Path(r"D:\2018-2019\Lake Superior Winter Deployment\WAV"),
    Path(r"D:\2018-2019\Lake Superior Summer Deployment\WAV"),
    Path(r"D:\2019-2020\WAV")
]

subset_size = 15
if os.path.exists(r"D:\POTENTIAL Fish sounds\random_files"):
    shutil.rmtree(r"D:\POTENTIAL Fish sounds\random_files")
    
# Output directory where selected files will be saved
output_root = Path(r"D:\POTENTIAL Fish sounds\random_files")
output_root.mkdir(exist_ok=True)

count = 0
for root in roots:
    for folder in root.iterdir():
        if folder.is_dir():
            files = [f for f in folder.iterdir() if f.is_file()]

            if len(files) == 0:
                print(f"No files in {folder}")
                continue

            chosen = random.sample(files, min(subset_size, len(files)))
            count += 1
            print(f"{folder} → selected {len(chosen)} files")

            for f in chosen:
                data, sr = sf.read(f)
                out_path = output_root / f.name
                sf.write(out_path, data, sr)
print(f"Total number of files: {count*subset_size}")

D:\2018-2019\Lake Superior Winter Deployment\WAV\201811 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201812 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201901 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201902 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201903 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201904 → selected 15 files
D:\2018-2019\Lake Superior Winter Deployment\WAV\201905 → selected 15 files
D:\2018-2019\Lake Superior Summer Deployment\WAV\052019 → selected 15 files
D:\2018-2019\Lake Superior Summer Deployment\WAV\062019 → selected 15 files
D:\2018-2019\Lake Superior Summer Deployment\WAV\072019 → selected 15 files
D:\2018-2019\Lake Superior Summer Deployment\WAV\082019 → selected 15 files
D:\2019-2020\WAV\201911 → selected 15 files
D:\2019-2020\WAV\201912 → selected 15 files
D:\2019-2020\WAV\202001 → selected 15 files
D:\2019-2020\WAV\202002 → select

In [7]:
def load_and_resample(path, target_sr=2048):
    y, sr = librosa.load(path, sr = target_sr)
    return y, target_sr

def spectrogram_encode(y, sr, n_fft=1024, hop_length=512):
    S = np.abs(librosa.stft(
        y,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        window='hann',
        center=True,
        pad_mode='constant'
    ))
    return np.log1p(S).T  # log compression

def extract_intervals(trigger_bool):
    """Return list of (start_frame, end_frame) for each True segment."""
    intervals = []
    in_event = False
    start = None

    for i, val in enumerate(trigger_bool):
        if val and not in_event:
            in_event = True
            start = i
        elif not val and in_event:
            in_event = False
            intervals.append((start, i - 1))

    # If the last segment ends at the final frame
    if in_event:
        intervals.append((start, len(trigger_bool) - 1))

    return intervals
def frame_to_seconds(frame, sr=1024, hop_length=128, trim_frames=60):
    return (frame + trim_frames) * hop_length / sr

In [16]:
if os.path.exists("random_snips"):
    shutil.rmtree("random_snips")

os.makedirs("random_snips", exist_ok=True)
# os.makedirs(f"random_snips/a", exist_ok=True)
# os.makedirs(f"random_snips/b", exist_ok=True)
# os.makedirs(f"random_snips/c", exist_ok=True)
# os.makedirs(f"random_snips/d", exist_ok=True)
# os.makedirs(f"random_snips/e", exist_ok=True)
os.makedirs(f"random_snips/z", exist_ok=True)

#load one for demo
sr = 1024
hop_length = 128

trig = 160

all_trigger_rows = []
file_paths = [f for f in output_root.iterdir() if f.is_file()]
for fname in file_paths: 
    path = Path(fname) 
    y, _ = load_and_resample(str(path), target_sr=sr)
    
    y_reduced = nr.reduce_noise(y=y, sr=sr, time_mask_smooth_ms = 320, prop_decrease = 0.20)
    X = spectrogram_encode(y_reduced, sr, hop_length = hop_length, n_fft = 256) #75% overlap, can play with it
    
    #throw out first 60 frames to avoid startup noise
    X = X[60:,:]


    # ---------------------
    # 5b
    # ---------------------
    #energy vs time
    energy = np.sum(X**2, axis=1)
    
    # Optional: normalize for nicer plotting
    energy_norm = energy / (np.max(energy) + 1e-12)
    

    # --------------------
    # 5c
    # --------------------
    high_energy_threshold = np.percentile(energy, 90)
    low_energy_frames = energy < high_energy_threshold
    X_whitened = X / (np.median(X[low_energy_frames], axis=0) + 1e-10)  # add small value to avoid division by zero

    log_whitened = np.log2(X_whitened + 1e-12)
    
    # Compute contrast limits
    vmin = np.percentile(log_whitened, 5)
    vmax = np.percentile(log_whitened, 95)
    

    # ---------------------
    # 5d
    # ---------------------
    #energy vs time
    energy = np.sum(X_whitened**2, axis=1)
  

    # -----------------------
    # 5e 
    # -----------------------
    X_whitened_normalized = X_whitened / (np.max(X_whitened) + 1e-10)  # normalize to [0, 1]
    def power_law_transform(X, window = 20, alpha=1.0):
        #create running average of the spectrum
        X_avg = np.zeros_like(X)
        for i in range(X.shape[0]): #use wraparound or padding for edge effects
            start = max(0, i - window // 2)
            end = min(X.shape[0], i + window // 2 + 1)
            X_avg[i] = np.median(X[start:end], axis=0)
        #apply power law transform
        X_avg = np.maximum(X_avg, np.percentile(X_avg, 5))
        X_power_law = X / (X_avg + 1e-10) ** alpha  # add small value to avoid division by zero
        return X_power_law

    X_pl = power_law_transform(X_whitened, alpha=0.5)
    log_pl = np.log2(X_pl + 1e-12)
    
    
    energy_running_sum = (np.sum(X_whitened[:-10], axis=1)) # average energy per frame

    
    new_energy = scipy.signal.medfilt(energy_running_sum, 7)

    trigger = new_energy > trig
    intervals = extract_intervals(trigger)
    rows = []
    for start_f, end_f in intervals:
        rows.append({
            "Filename": fname,
            "Begin_Frame": start_f,
            "End_Frame": end_f,
            "Begin_Second": frame_to_seconds(start_f, sr, hop_length, 60),
            "End_Second": frame_to_seconds(end_f, sr, hop_length, 60)
        })

    
    target_seconds = 4.0
    frames_per_second = sr / hop_length
    target_frames = int(target_seconds * frames_per_second)
    num_frames_total = log_pl.shape[0]
    
    for start_f, end_f in intervals:
            interval_length = end_f - start_f
            if interval_length > target_frames:
                num_chunks = math.ceil(interval_length / target_frames)
                for i in range(int(num_chunks)):
                    chunk_st = start_f + i * target_frames
                    chunk_end = chunk_st + target_frames
                    # yield or save this chunk
                    chunk_st = int(round(chunk_st))
                    chunk_end = int(round(chunk_end))
                
                    if chunk_end > end_f:
                        continue

                    snip = log_pl[int(chunk_st):int(chunk_end), :]
                    # do whatever you normally do with snip
            
                    snip_norm = (snip - snip.min()) / (snip.max() - snip.min() + 1e-12)
                    cat = np.random.choice(["a", "b", "c", "d", "e"])[0] #rand assigning category                            
                    out_name = f"{Path(fname).stem}_{chunk_st}_{chunk_end}.png"
                    out_path = f"random_snips/z/{out_name}"
                    cv2.imwrite(out_path, (snip_norm * 255).astype(np.uint8))
        
                continue  # skip padding logic
            needed_padding = (target_frames - interval_length)/2
            if (end_f + needed_padding <= num_frames_total) and (start_f - needed_padding >= 0):
                new_st_f = start_f - needed_padding
                new_end_f = end_f + needed_padding
            elif (end_f + needed_padding <= num_frames_total) and (start_f - needed_padding < 0):
                new_st_f = 0
                new_end_f = end_f + needed_padding + np.abs(start_f - needed_padding)
            elif (end_f + needed_padding > num_frames_total) and (start_f - needed_padding >= 0):
                new_st_f = start_f - needed_padding - (end_f + needed_padding - log_pl.shape[0])
                new_end_f = log_pl.shape[0]
            else: 
                print(f"Index out of bounds for file {fname}")
                continue
    
            new_st_f = int(round(new_st_f))
            new_end_f = int(round(new_end_f))
        
            new_st_f = max(0, new_st_f)
            new_end_f = min(num_frames_total, new_end_f)
        
            snip = log_pl[new_st_f:new_end_f, :]
    
            snip_norm = (snip - snip.min()) / (snip.max() - snip.min() + 1e-12)
            cat = np.random.choice(["a", "b", "c", "d", "e"])[0] #rand assigning category                            
            out_name = f"{Path(fname).stem}_{new_st_f}_{new_end_f}.png"
            out_path = f"random_snips/z/{out_name}"
            cv2.imwrite(out_path, (snip_norm * 255).astype(np.uint8))
    
    df_events = pd.DataFrame(rows)

    
    # Save it for later
    all_trigger_rows.append(df_events)


df_all = pd.concat(all_trigger_rows, ignore_index=True)

In [9]:
df_all

,Filename,Begin_Frame,End_Frame,Begin_Second,End_Second
0,D:\POTENTIAL Fish sounds\random_files\12082217...,18,26,9.750,10.750
1,D:\POTENTIAL Fish sounds\random_files\12082217...,45,71,13.125,16.375
2,D:\POTENTIAL Fish sounds\random_files\12082217...,77,140,17.125,25.000
3,D:\POTENTIAL Fish sounds\random_files\12082217...,148,151,26.000,26.375
4,D:\POTENTIAL Fish sounds\random_files\12082217...,184,193,30.500,31.625
...,...,...,...,...,...
1161,D:\POTENTIAL Fish sounds\random_files\67163344...,837,850,112.125,113.750
1162,D:\POTENTIAL Fish sounds\random_files\67163344...,63,70,15.375,16.250
1163,D:\POTENTIAL Fish sounds\random_files\67163344...,574,577,79.250,79.625
1164,D:\POTENTIAL Fish sounds\random_files\67163344...,682,688,92.750,93.500
